### Spatial niches: ligand-receptor pairs, gradients and biological programs

Follows `squid01_analyze_visium_HE_data.ipynb` (mouse brain, Visium H&E, `visium_hne.h5ad`).

For a central spot (or the core of a region):
1. map the k ≤ 20 nearest spots (Visium hex grid: 6 + 12 spots in rings 1-2),
2. score every ligand-receptor (LR) pair in both directions, **center → neighbors** and **neighbors → center**, against 2,000 random niches of the same tissue (z, empirical p, FDR),
3. measure ligand and receptor **gradients** along 3-6 collinear spots, from the center towards the best partner spot,
4. infer **biological programs** (50 functional programs, Reactome-summarised) from niche program activity + programs containing the significant L/R genes.

Caveat: a Visium spot is 55 µm (≈1-10 cells), so these are spot-level niches; gradients over 3-6 points are descriptive.

In [ ]:
import numpy as np
import pandas as pd
import os, sys

import anndata as ad
import scanpy as sc
import squidpy as sq
from pathlib import Path

ROOT0 = Path("/home/flavio/uv/gwas/")
ROOT_SRC = ROOT0 / "src"
sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv
from libs.matplotlib_figure_style import *

import matplotlib.pyplot as plt

root0 = Path('/home/flavio/uv/squidpy/')
root_data = root0 / 'data'
root_figure = root0 / 'figures'
root_result = root0 / 'results'

# folder with niche_lr.py, programs.py, lr_mouseconsensus.csv
ROOT_NICHE = root0 / 'src' / 'spatial_niche_lr'
sys.path.append(str(ROOT_NICHE))

import niche_lr as nl
import programs as pg

In [ ]:
filename_vis = root_data / "visium_hne.h5ad"
adata = ad.read_h5ad(filename_vis)
adata

### Expression layer

The niche scores need **log-normalised, non-negative** values. In this object `adata.X` is log-normalised and `.raw` holds counts, so `layer=None` (use `X`).

In [ ]:
X = adata.X
print(type(X), X.shape, "min", X.min(), "max", X.max())

coords = adata.obsm["spatial"]
spacing = nl.spot_spacing(coords)          # ~137 px centre-to-centre
um_per_px = 100 / spacing                  # Visium: 100 um centre-to-centre
print(f"spot spacing = {spacing:.1f} px  ->  {um_per_px:.3f} um/px")

### Programs (human catalogue -> mouse symbols)

`to_mouse` capitalises human symbols and applies explicit orthologs (HLA -> H2, FCGR3A -> Fcgr4, ...); genes absent from the data are dropped and poorly covered programs are reported.

In [ ]:
programs_m = pg.to_mouse(pg.PROGRAMS, adata.var_names)
states_m   = pg.to_mouse(pg.CELL_STATES, adata.var_names)

df_prog = pd.DataFrame([(c, p, len(g)) for c, d in programs_m.items() for p, g in d.items()],
                       columns=["category", "program", "n_genes_in_data"])
print(df_prog.shape)
df_prog.sort_values("n_genes_in_data").head(10)

### Reactome: every pathway -> program (or background class)

`reactome_pathways.tsv` has names only (no genes), so this is the name-based summary of all Reactome pathways into the 50 programs.
Pathways that are disease variants, pathogens or drugs are *excluded*; housekeeping biology goes to *context* classes (transcription, translation, transport, GPCR, ...).
With `ReactomePathways.gmt` (gene members) use `pg.expand_with_reactome(pg.load_gmt(path))` to build the complete gene sets.

In [ ]:
fname_reac = root_data / "reactome_pathways.tsv"

df_reac = pg.load_reactome_tsv(fname_reac, species="Mus musculus")
df_cls  = pg.classify_reactome(df_reac)
df_sum  = pg.summarize_reactome(df_cls)

print(df_cls.drop_duplicates("pathway").level.value_counts())

fname = "reactome_mouse_pathway_to_program.tsv"
if not (root_result / fname).exists():
    pdwritecsv(df_cls, fname, root_result, verbose=True)

df_sum[df_sum.level == "program"].head(20)

### Choose the center

Options: `barcode=...`, `xy=(x, y)`, `mask=<bool array or obs column>`, `program="synaptic_neurotransmission"` (hotspot), or one center per anatomical cluster (`centers_by_group`: the most interior spot of each cluster).

In [ ]:
centers = nl.centers_by_group(adata, "cluster")
centers

In [ ]:
cluster_name = "Hippocampus"
center = centers[cluster_name]

# alternatives:
# center = nl.select_center(adata, program="synaptic_neurotransmission", programs=programs_m)
# center = nl.select_center(adata, barcode="AAACAAGTATCTCCCA-1")

lr_m = nl.load_lr("mouseconsensus")
print(lr_m.shape, adata.obs_names[center], adata.obs.loc[adata.obs_names[center], "cluster"])

### Run the niche analysis

In [ ]:
# background: whole tissue (None) -> "what is special about this niche";
#             other clusters        -> "this region vs the rest"
bg_mask = (adata.obs["cluster"] != cluster_name).values
# bg_mask = None

res = nl.analyze_niche(
    adata, center, k=20,
    center_pool=6,         # center + ring 1 pooled into a central region (damps single-spot noise);
                           # partners = the remaining 14 spots (ring 2 + 2 of ring 3). 0 = single spot
    lr=lr_m, programs=programs_m,
    bg_mask=bg_mask,
    layer=None,            # adata.X is log-normalised
    fdr=0.1,
    n_top_gradients=15,
    n_line_cells=6,        # 3-6 collinear spots
    um_per_unit=um_per_px, # gradients per um
    n_background=2000,
)
res.neighbors.head(8)

### Neighbours: cluster composition

In [ ]:
nb = res.neighbors.copy()
nb["cluster"] = adata.obs["cluster"].values[nb.idx]
nb.groupby(["ring", "cluster"], observed=True).size().unstack(fill_value=0)

### Top LR pairs (both directions)

`center->neighbors`: ligand in the center, receptor in the neighbours. `neighbors->center`: the reverse.
`z` = niche score vs. random niches. `fdr` uses the empirical p, switching to the Gaussian tail of z when fewer than 5 background niches reach the observed score (the empirical p has a floor of 1/(n_background+1)); `fdr_emp` is the purely empirical version.

In [ ]:
cols = ["ligand", "receptor", "direction", "score", "z", "fdr", "fdr_emp",
        "ligand_sender", "receptor_receiver", "ligand_programs", "receptor_programs"]

df_lr = res.lr

fname = f"niche_lr_{cluster_name}.tsv"
if not (root_result / fname).exists():
    pdwritecsv(df_lr, fname, root_result, verbose=True)

df_lr[cols].head(25)

### Inferred biological functions

`z` = niche program activity vs. random niches; `lr_support` = Σ -log10(p) of significant LRs whose L or R is in the program; `score` combines both ranks.

In [ ]:
df_fun = res.functions

fname = f"niche_functions_{cluster_name}.tsv"
if not (root_result / fname).exists():
    pdwritecsv(df_fun, fname, root_result, verbose=True)

df_fun[["category", "program", "z", "percentile", "lr_support", "n_lr", "top_lr", "score"]].head(15)

### Cell states in the niche (identity markers, activity only)

In [ ]:
df_states = nl.niche_program_activity(adata, center, res.neighbors, states_m, n_background=2000, bg_mask=bg_mask)
df_states.head(10)

### Gradients along collinear spots

A slope counts as a trend only if |Spearman ρ| ≥ 0.6 (`pattern`: source→sink, co-decreasing, ligand/receptor gradient only, flat).
`consistent_with_direction`: center→neighbors → ligand highest at the center and decreasing, receptor expressed downstream; neighbors→center → receptor highest at the center, ligand rising towards the sender.

In [ ]:
df_grad = res.gradients
df_grad[["key", "n_cells", "L_slope", "L_rho", "R_slope", "R_rho", "LR_corr",
         "pattern", "consistent_with_direction", "z", "fdr"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
nl.plot_niche(adata, res, ax=ax)

filename = root_figure / f"niche_map_{cluster_name}.png"
if not filename.exists():
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")
plt.show()

In [ ]:
keys = df_grad.sort_values(["consistent_with_direction", "z"], ascending=False).key.head(6).tolist()
fig = nl.plot_gradients(res, keys)

filename = root_figure / f"niche_gradients_{cluster_name}.png"
if not filename.exists():
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")
plt.show()

### Cross-check with LIANA+ bivariate (from squid01)

Local cosine score of the same LR at the center and its neighbours, and global rank by Moran's R.

In [ ]:
filename_lr = root_data / "visium_hne_liana_bivariate.h5ad"
if filename_lr.exists():
    lrdata = ad.read_h5ad(filename_lr)
    df_chk = nl.liana_local_check(lrdata, res, n=25)
    display(df_chk)

### All clusters: program x niche matrix

In [ ]:
fname = "niche_functions_by_cluster.tsv"
force = False

if (root_result / fname).exists() and not force:
    df_mat = pdreadcsv(fname, root_result).set_index("program")
else:
    df_mat, results = nl.scan_niches(adata, centers, value="z", exclude_own_group="cluster",
                                     center_pool=6, lr=lr_m, programs=programs_m,
                                     um_per_unit=um_per_px, n_background=1000, n_top_gradients=10)
    pdwritecsv(df_mat.reset_index(), fname, root_result, verbose=True)

df_mat.round(1).head()

In [ ]:
import seaborn as sns

top = df_mat.loc[df_mat.abs().max(1).sort_values(ascending=False).index[:30]]
g = sns.clustermap(top, cmap="coolwarm", center=0, figsize=(12, 10), vmin=-5, vmax=5,
                   cbar_kws={"label": "niche activity z"})

filename = root_figure / "niche_programs_by_cluster.png"
if not filename.exists():
    g.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")
plt.show()